# Live Prediction

In [ ]:
# ================================
# FISH APP PROJECT - SYNTHETIC DATA GENERATION, MODEL TRAINING, AND LIVE PREDICTION
# ================================

import numpy as np
import pandas as pd
import math
import random
import requests
from datetime import datetime, timedelta

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


# ----------------
# 1. DEFINE COLUMNS
# ----------------
columns = [
    "user_id",
    "date_time",
    "month",
    "latitude",
    "longitude",
    "species",
    "fish_count",
    "avg_size",
    "temperature",
    "weather",
    "wind_speed",
    "time_of_day",
    "season",
    "tide_level",
    "tide_stage",
    "noaa_station_id",
    "success"
]


# ----------------
# 2. LIVE WEATHER AND TIDE FUNCTION
# ----------------
def get_live_weather(lat, lon):
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}&longitude={lon}"
        f"&current=temperature_2m,wind_speed_10m,weather_code"
        f"&temperature_unit=fahrenheit"
        f"&wind_speed_unit=mph"
    )

    response = requests.get(url, timeout=20)
    response.raise_for_status()
    data = response.json()["current"]

    temperature = data["temperature_2m"]
    wind_speed = data["wind_speed_10m"]
    weather_code = data["weather_code"]

    weather_map = {
        0: "sunny",
        1: "mostly_clear",
        2: "partly_cloudy",
        3: "cloudy",
        61: "rainy",
        63: "rainy",
        65: "rainy"
    }

    weather = weather_map.get(weather_code, "unknown")

    return {
        "temperature": temperature,
        "wind_speed": wind_speed,
        "weather": weather
    }

def haversine_miles(lat1, lon1, lat2, lon2):
    R = 3958.8  # Earth radius in miles

    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.asin(math.sqrt(a))

    return R * c


def get_nearest_noaa_station(lat, lon):
    """
    Finds the nearest active NOAA CO-OPS water level station.
    """
    url = "https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations.json?type=waterlevels"
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    data = response.json()
    stations = data["stations"]

    closest_station = None
    closest_distance = float("inf")

    for station in stations:
        station_lat = station["lat"]
        station_lon = station["lng"]

        distance = haversine_miles(lat, lon, station_lat, station_lon)

        if distance < closest_distance:
            closest_distance = distance
            closest_station = {
                "id": station["id"],
                "name": station["name"],
                "lat": station_lat,
                "lon": station_lon,
                "distance_miles": distance
            }

    return closest_station


def get_latest_tide_level(station_id):
    """
    Gets the latest observed water level from NOAA CO-OPS.
    Uses product=water_level and date=latest.
    """
    url = (
        "https://api.tidesandcurrents.noaa.gov/api/prod/datagetter"
        f"?product=water_level"
        f"&application=fish_app"
        f"&date=latest"
        f"&station={station_id}"
        f"&datum=MLLW"
        f"&units=english"
        f"&time_zone=lst_ldt"
        f"&format=json"
    )

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    data = response.json()

    # NOAA usually returns a "data" list for this product
    latest = data["data"][0]

    return {
        "t": latest["t"],
        "v": float(latest["v"])
    }


def get_today_high_low_predictions(station_id):
    """
    Gets today's high/low tide predictions from NOAA CO-OPS.
    Uses product=predictions and interval=hilo.
    """
    url = (
        "https://api.tidesandcurrents.noaa.gov/api/prod/datagetter"
        f"?product=predictions"
        f"&application=fish_app"
        f"&date=today"
        f"&station={station_id}"
        f"&datum=MLLW"
        f"&interval=hilo"
        f"&units=english"
        f"&time_zone=lst_ldt"
        f"&format=json"
    )

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    data = response.json()

    # NOAA returns predictions in a "predictions" list for this product
    preds = data["predictions"]

    parsed = []
    for item in preds:
        parsed.append({
            "time": datetime.strptime(item["t"], "%Y-%m-%d %H:%M"),
            "type": item["type"],   # H or L
            "value": float(item["v"])
        })

    return parsed


def infer_tide_stage(predictions, current_time=None):
    """
    Infers whether the tide is rising or falling based on the
    previous and next high/low tide prediction.
    """
    if current_time is None:
        current_time = datetime.now()

    previous_event = None
    next_event = None

    for event in predictions:
        if event["time"] <= current_time:
            previous_event = event
        elif event["time"] > current_time and next_event is None:
            next_event = event

    if previous_event is None or next_event is None:
        return "unknown"

    if previous_event["type"] == "L" and next_event["type"] == "H":
        return "rising"
    elif previous_event["type"] == "H" and next_event["type"] == "L":
        return "falling"
    else:
        return "unknown"


def get_realtime_tide(lat, lon):
    """
    Main helper:
    1. find nearest NOAA water-level station
    2. get latest observed tide level
    3. get today's high/low predictions
    4. infer tide stage
    """
    station = get_nearest_noaa_station(lat, lon)
    latest_level = get_latest_tide_level(station["id"])
    predictions = get_today_high_low_predictions(station["id"])
    tide_stage = infer_tide_stage(predictions)

    return {
        "noaa_station_id": station["id"],
        "noaa_station_name": station["name"],
        "station_distance_miles": station["distance_miles"],
        "tide_level": latest_level["v"],
        "tide_time": latest_level["t"],
        "tide_stage": tide_stage
    }
LIVE_LAT = 34.2257
LIVE_LON = -77.9447

try:
    tide_snapshot = get_realtime_tide(LIVE_LAT, LIVE_LON)
    print("Realtime tide snapshot:")
    print(tide_snapshot)
except Exception as e:
    print("Could not retrieve tide data.")
    print("Error:", e)

def apply_species_logic(species, success_prob, month, season, temperature, wind_speed, time_of_day, weather):
    """
    Domain-informed adjustments for NC coastal species.
    These are still simulated weights, but they are based on
    seasonality / life-history patterns described by NC DEQ materials.
    """

    # ----------------
    # RED DRUM
    # NC-informed idea:
    # Treat spring/fall as stronger inshore periods and morning/evening as slightly better.
    # ----------------
    if species == "red_drum":
        if season in ["spring", "fall"]:
            success_prob += 0.12
        if time_of_day in ["morning", "evening"]:
            success_prob += 0.05
        if wind_speed < 10:
            success_prob += 0.04

    # ----------------
    # FLOUNDER (Southern Flounder)
    # NC DEQ: estuarine life, offshore spawning in fall/winter
    # Stronger fall signal, moderate late-summer/early-fall migration signal
    # ----------------
    elif species == "flounder":
        if month in [9, 10, 11]:
            success_prob += 0.18
        elif month in [8, 12]:
            success_prob += 0.08

        if wind_speed < 8:
            success_prob += 0.05

        if time_of_day in ["morning", "evening"]:
            success_prob += 0.03

    # ----------------
    # WEAKFISH (gray trout)
    # NC DEQ/FMP update: peak spawning around April-May, spring/summer spawning,
    # leave estuaries in fall as temps drop
    # ----------------
    elif species == "weakfish":
        if month in [4, 5]:
            success_prob += 0.18
        elif month in [3, 6, 7]:
            success_prob += 0.10

        if 58 <= temperature <= 72:
            success_prob += 0.06

        if time_of_day == "evening":
            success_prob += 0.04

    # ----------------
    # SPOTTED SEATROUT (speckled trout)
    # NC DEQ: estuarine species, vulnerable to winter cold stuns,
    # so avoid boosting very cold conditions; favor moderate temps
    # ----------------
    elif species == "spotted_seatrout":
        if month in [4, 5, 6, 9, 10]:
            success_prob += 0.12

        if 58 <= temperature <= 72:
            success_prob += 0.10
        elif temperature < 50:
            success_prob -= 0.18

        if time_of_day in ["morning", "evening"]:
            success_prob += 0.04

        if weather in ["cloudy", "partly_cloudy"]:
            success_prob += 0.03

    # ----------------
    # STRIPED BASS
    # NC materials support temperature/habitat sensitivity;
    # use cooler months / cooler temperatures as a stronger signal
    # ----------------
    elif species == "striped_bass":
        if month in [10, 11, 12, 1, 2, 3]:
            success_prob += 0.14

        if temperature < 65:
            success_prob += 0.10
        elif temperature > 78:
            success_prob -= 0.10

        if time_of_day == "morning":
            success_prob += 0.03

    return success_prob

# ----------------
# 3. SYNTHETIC DATA GENERATOR 
# ----------------
def generate_synthetic_data(n=2000):
    data = []

    species_list = [
        "red_drum",
        "flounder",
        "weakfish",
        "spotted_seatrout",
        "striped_bass"
    ]

    weather_list = ["sunny", "cloudy", "rainy", "partly_cloudy"]
    time_list = ["morning", "afternoon", "evening"]

    for i in range(n):
        user_id = random.randint(1, 50)

        date_time = datetime.now() - timedelta(days=np.random.randint(0, 365))
        month = date_time.month

        latitude = np.random.uniform(33, 37)
        longitude = np.random.uniform(-80, -75)

        species = random.choice(species_list)
        temperature = np.random.uniform(40, 90)
        weather = random.choice(weather_list)
        wind_speed = np.random.uniform(0, 20)
        time_of_day = random.choice(time_list)

        # derive season from month
        if month in [12, 1, 2]:
            season = "winter"
        elif month in [3, 4, 5]:
            season = "spring"
        elif month in [6, 7, 8]:
            season = "summer"
        else:
            season = "fall"

        # base probability
        success_prob = np.random.uniform(0.1, 0.4)

        # general fishing-condition logic
        if 58 <= temperature <= 78:
            success_prob += 0.10
        if wind_speed < 10:
            success_prob += 0.10
        if time_of_day == "morning":
            success_prob += 0.05
        if weather in ["cloudy", "partly_cloudy"]:
            success_prob += 0.05
        
        # simulate multiple fishing zones

        # strong hotspot
        if 35.5 < latitude < 36.2 and -78 < longitude < -76:
            success_prob += 0.12

        # moderate hotspot
        elif 34.5 < latitude < 35.5 and -79 < longitude < -77:
            success_prob += 0.07

        # poor zone
        elif latitude < 34:
            success_prob -= 0.08

        

        # species-specific adjustment
        success_prob = apply_species_logic(
            species=species,
            success_prob=success_prob,
            month=month,
            season=season,
            temperature=temperature,
            wind_speed=wind_speed,
            time_of_day=time_of_day,
            weather=weather
        )

        # noise so the model is not unrealistically perfect
        success_prob += np.random.normal(0, 0.20)
        success_prob = np.clip(success_prob, 0.03, 0.92)

        success = int(np.random.rand() < success_prob)

        # ADD NOISE
        if np.random.rand() < 0.10:
            success = 1

        if np.random.rand() < 0.10:
            success = 0

        fish_count = np.random.randint(1, 5) if success else 0
        avg_size = np.random.uniform(10, 30) if success else 0
        # synthetic tide placeholders for training data
        synthetic_tide_stage = random.choice(["rising", "falling"])
        synthetic_tide_level = np.random.uniform(-1.0, 5.0)
        # Tide effect (REALISTIC RULE)
        if synthetic_tide_stage == "rising":
            success_prob += 0.08

        if synthetic_tide_stage == "falling":
            success_prob += 0.03
        synthetic_station_id = f"station_{random.randint(1, 8)}"

        row = [
            user_id,
            date_time,
            month,
            latitude,
            longitude,
            species,
            fish_count,
            avg_size,
            temperature,
            weather,
            wind_speed,
            time_of_day,
            season,
            synthetic_tide_stage,
            synthetic_tide_level,
            synthetic_station_id,
            success
        ]

        data.append(row)

    return pd.DataFrame(data, columns=columns)


# ----------------
# 4. CREATE DATASET
# ----------------
df = generate_synthetic_data(2000)

print("Dataset created")
print(df.head())
print("\nSuccess counts:")
print(df["success"].value_counts())


# ----------------
# 5. MODEL PREP
# ----------------
X = df.drop(columns=["success", "date_time", "fish_count", "avg_size"])
y = df["success"]

X = pd.get_dummies(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# ----------------
# 6. TRAIN MODEL
# ----------------

# ----------------
# TRAIN LOGISTIC REGRESSION
# ----------------
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print("\nLogistic Regression Accuracy:", round(accuracy, 4))


# ----------------
# TRAIN RANDOM FOREST
# ----------------
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_accuracy = rf_model.score(X_test, y_test)
print("Random Forest Accuracy:", round(rf_accuracy, 4))

# ----------------
# TRAIN XGBOOST
# ----------------
xgb_model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)

xgb_accuracy = xgb_model.score(X_test, y_test)
print("\nXGBoost Accuracy:", round(xgb_accuracy, 4))

def predict_fishing_success(model, lat, lon, species, time_of_day, season, user_id=999):
    live_weather = get_live_weather(lat, lon)
    live_tide = get_realtime_tide(lat, lon)

    sample = pd.DataFrame([{
        "user_id": user_id,
        "month": datetime.now().month,
        "latitude": lat,
        "longitude": lon,
        "species": species,
        "temperature": live_weather["temperature"],
        "weather": live_weather["weather"],
        "wind_speed": live_weather["wind_speed"],
        "time_of_day": time_of_day,
        "season": season,
        "tide_level": live_tide["tide_level"],
        "tide_stage": live_tide["tide_stage"],
        "noaa_station_id": live_tide["noaa_station_id"]
    }])

    sample_encoded = pd.get_dummies(sample)
    sample_encoded = sample_encoded.reindex(columns=X.columns, fill_value=0)

    prob = model.predict_proba(sample_encoded)[0][1]
    pred = model.predict(sample_encoded)[0]

    return {
        "prediction": int(pred),
        "success_probability": float(prob),
        "live_weather": live_weather,
        "live_tide": live_tide
    }

prediction_result = predict_fishing_success(
    model=xgb_model,
    lat=34.2257,
    lon=-77.9447,
    species="red_drum",
    time_of_day="morning",
    season="spring"
)

print("\nLive prediction result:")
print(prediction_result)

# ----------------
# 7. FEATURE IMPORTANCE
# ----------------
lr_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.coef_[0]
}).sort_values(by="importance", ascending=False)

print("\nTop Features:")
print(lr_importance.head(10))

rf_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_
}).sort_values(by="importance", ascending=False)

print("\nRandom Forest Top Features:")
print(rf_importance.head(10))

xgb_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": xgb_model.feature_importances_
}).sort_values(by="importance", ascending=False)

print("\nXGBoost Top Features:")
print(xgb_importance.head(10))


# ----------------
# 8. INSIGHTS
# ----------------
print("\nSuccess by species:")
print(df.groupby("species")["success"].mean())

print("\nSuccess by time of day:")
print(df.groupby("time_of_day")["success"].mean())

print("\nSuccess by weather:")
print(df.groupby("weather")["success"].mean())

Realtime tide snapshot:
{'noaa_station_id': '8658120', 'noaa_station_name': 'Wilmington', 'station_distance_miles': 0.4961408008596001, 'tide_level': 3.924, 'tide_time': '2026-04-19 09:24', 'tide_stage': 'rising'}
Dataset created
   user_id                  date_time  month   latitude  longitude   species  \
0       22 2026-03-30 09:32:20.207186      3  35.506501 -78.482106  flounder   
1       19 2025-07-06 09:32:20.207427      7  36.214646 -78.210445  flounder   
2       16 2025-11-22 09:32:20.207485     11  35.937567 -75.631966  weakfish   
3        6 2025-11-18 09:32:20.207529     11  33.269124 -76.661079  weakfish   
4       22 2025-09-28 09:32:20.207568      9  34.159933 -76.437470  flounder   

   fish_count   avg_size  temperature weather  wind_speed time_of_day  season  \
0           1  21.005357    77.152857   sunny   19.839464     morning  spring   
1           0   0.000000    83.095388   rainy   18.340703     morning  summer   
2           0   0.000000    84.236342  cloudy 

In [2]:
print("\nModel Comparison")
print("----------------")
print("Logistic Regression:", round(accuracy, 4))
print("Random Forest:      ", round(rf_accuracy, 4))
print("XGBoost:            ", round(xgb_accuracy, 4))


Model Comparison
----------------
Logistic Regression: 0.565
Random Forest:       0.585
XGBoost:             0.605


In [3]:
df.groupby("species")["success"].mean()

species
flounder            0.453540
red_drum            0.444759
spotted_seatrout    0.458234
striped_bass        0.468254
weakfish            0.439698
Name: success, dtype: float64

In [4]:
from sklearn.metrics import roc_auc_score

y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
roc = roc_auc_score(y_test, y_prob_xgb)

print("XGBoost ROC-AUC:", round(roc, 4))

XGBoost ROC-AUC: 0.6073


# Scoring

In [9]:
print("\nXGBoost Top Features:")
print(xgb_importance.head(10))


XGBoost Top Features:
                      feature  importance
31  noaa_station_id_station_7    0.046843
18        time_of_day_morning    0.041802
13      weather_partly_cloudy    0.036715
17        time_of_day_evening    0.036676
4                 temperature    0.035259
15              weather_sunny    0.035080
16      time_of_day_afternoon    0.033832
2                    latitude    0.033580
19                season_fall    0.033447
3                   longitude    0.032401


In [15]:
def get_fishing_score(model, lat, lon, species, time_of_day, season, user_id=999):
    # get live weather and live tide
    live_weather = get_live_weather(lat, lon)
    live_tide = get_realtime_tide(lat, lon)

    # build one input row
    sample = pd.DataFrame([{
        "user_id": user_id,
        "month": datetime.now().month,
        "latitude": lat,
        "longitude": lon,
        "species": species,
        "temperature": live_weather["temperature"],
        "weather": live_weather["weather"],
        "wind_speed": live_weather["wind_speed"],
        "time_of_day": time_of_day,
        "season": season,
        "tide_level": live_tide["tide_level"],
        "tide_stage": live_tide["tide_stage"],
        "noaa_station_id": live_tide["noaa_station_id"]
    }])

    # encode to match training data
    sample_encoded = pd.get_dummies(sample)
    sample_encoded = sample_encoded.reindex(columns=X.columns, fill_value=0)

    # model probability of success
    prob_success = model.predict_proba(sample_encoded)[0][1]

    # convert to 0-100 score
    fishing_score = round(prob_success * 100)

    # label the score
    if fishing_score < 40:
        rating = "Poor"
    elif fishing_score < 70:
        rating = "Fair"
    elif fishing_score < 85:
        rating = "Good"
    else:
        rating = "Excellent"

    # explanation
    explanation = []

    if live_weather["wind_speed"] < 10:
        explanation.append("low wind")
    if live_weather["weather"] in ["cloudy", "partly_cloudy", "mostly_clear"]:
        explanation.append("favorable sky conditions")
    if live_tide["tide_stage"] == "rising":
        explanation.append("rising tide")
    if time_of_day == "morning":
        explanation.append("morning conditions")

    if not explanation:
        explanation_text = "Conditions are mixed right now."
    else:
        explanation_text = "Score helped by: " + ", ".join(explanation) + "."

    return {
        "fishing_score": fishing_score,
        "rating": rating,
        "probability_of_success": round(prob_success * 100),
        "species": species,
        "time_of_day": time_of_day,
        "season": season,
        "live_weather": live_weather,
        "live_tide": live_tide,
        "explanation": explanation_text
    }

In [18]:
def display_fishing_score(score_result):
    rating = score_result["rating"]

    if rating == "Poor":
        emoji = "🔴"
    elif rating == "Fair":
        emoji = "🟡"
    elif rating == "Good":
        emoji = "🟢"
    else:
        emoji = "⭐"

    print("\n🎣 Fishing Conditions Report")
    print("-" * 35)

    print(f"Species: {score_result['species'].replace('_', ' ').title()}")
    print(f"Fishing Score: {score_result['fishing_score']}/100")
    print(f"Rating: {emoji} {rating}")
    print(f"Chance of Success: {round(score_result['probability_of_success'])}%")

    print("\n🌤 Current Conditions")
    print(f"Weather: {score_result['live_weather']['weather'].replace('_', ' ').title()}")
    print(f"Temperature: {score_result['live_weather']['temperature']}°F")
    print(f"Wind Speed: {score_result['live_weather']['wind_speed']} mph")

    print("\n🌊 Tide Conditions")
    print(f"Tide Stage: {score_result['live_tide']['tide_stage'].title()}")
    print(f"Tide Level: {score_result['live_tide']['tide_level']} ft")

    print("\n💡 Why This Score?")
    print(score_result['explanation'])

In [19]:
score_result = get_fishing_score(
    model=xgb_model,
    lat=34.2257,
    lon=-77.9447,
    species="red_drum",
    time_of_day="morning",
    season="spring"
)

display_fishing_score(score_result)


🎣 Fishing Conditions Report
-----------------------------------
Species: Red Drum
Fishing Score: 52/100
Rating: 🟡 Fair
Chance of Success: 52%

🌤 Current Conditions
Weather: Cloudy
Temperature: 78.5°F
Wind Speed: 10.5 mph

🌊 Tide Conditions
Tide Stage: Rising
Tide Level: 4.213 ft

💡 Why This Score?
Score helped by: favorable sky conditions, rising tide, morning conditions.
